# MODIS Aqua **diffuse attenuation Kd(490)** over DBO — streaming, EASE-Grid 2.0 North

Companion to the chl / bbp / SST / sea-ice DBO notebooks. Streams the **MODIS-Aqua Level-3 8-day
mapped diffuse attenuation coefficient at 490 nm (Kd_490, water clarity / light penetration)** (`MODISA_L3m_KD`, 4 km), grids each month onto the **same EASE-2 4 km DBO
grid**, and folds it into in-memory accumulators (nothing written but a checkpoint). Same through-time
analysis: per-box monthly + deseasonalized anomaly + Theil-Sen/AR(1) Mann-Kendall, region map, anomaly GIF, per-pixel trend map.

### Product
- **`MODISA_L3m_KD`** v2022.0 (concept_id `C3380709198-OB_CLOUD`), file pattern `*.8D.KD.Kd_490.4km.nc`
  -> **~1076 8-day granules, 2002-07 -> 2025-11**. Variable **`Kd_490`** (units m^-1).
- Kd(490) = how fast blue-green light is attenuated with depth; higher = more turbid / shallower euphotic zone (driven by chl, CDOM and sediment together). Inverse of water clarity.
- Strictly positive -> non-positive fills masked to NaN. Trends on **linear** deseasonalized anomalies (a log10 variant is noted in S8).

### Reach
- Ocean color needs open water + sunlight, so all DBO boxes get data in their open-water season; high-latitude
  boxes (DBO3-8) are **summer-only** (polar night + winter ice -> NaN months; e.g. December is empty over DBO).

## 1. Imports

In [1]:
import time as _time, warnings, gc
from pathlib import Path
from datetime import date

import numpy as np, pandas as pd, xarray as xr
import earthaccess
from pyresample import AreaDefinition, geometry
from pyresample.kd_tree import resample_nearest
from matplotlib.path import Path as MplPath
import matplotlib.pyplot as plt, matplotlib.dates as mdates
import cartopy.crs as ccrs, cartopy.feature as cfeature
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from scipy.stats import theilslopes, kendalltau, norm
warnings.filterwarnings("ignore")
print("imports OK")

imports OK


## 2. Configuration - product, EASE-2 grid, DBO boxes

In [2]:
# >>> confirm with S3 (search) and S3b (inspect) <<<
OC_SHORT   = "MODISA_L3m_KD"
OC_CONCEPT = "C3380709198-OB_CLOUD"
GRANULE_PAT= "*.8D.KD.Kd_490.4km.nc"
VAR        = "Kd_490"                       # diffuse attenuation coefficient at 490 nm (Kd_490, water clarity / light penetration) (m^-1)
LABEL      = "Kd(490)"                     # short axis label
UNITS      = "m$^{-1}$"
LATN, LONN = "lat", "lon"

START_DATE = date(2002, 7, 1)                     # MODIS-Aqua ocean-color record start
END_DATE   = date.today()
LAT_MIN, LAT_MAX = 45, 90
ROI = 6_000                                       # nearest-neighbour radius (m); L3m is 4 km
N_WORKERS = 8
DBO_BBOX = (-180.0, 50.0, -124.0, 74.0)

dbo_area = AreaDefinition(
    area_id="ease2_north_dbo_4km",
    description="EASE-Grid 2.0 North subset: Aleutians -> Beaufort (DBO1-8), 4 km",
    proj_id="EPSG:6931",
    projection={"proj":"laea","lat_0":90,"lon_0":0,"x_0":0,"y_0":0,"datum":"WGS84","units":"m"},
    width=906, height=845, area_extent=(-3_624_000, 992_000, 0, 4_372_000))

CHECKPOINT = True
CKPT_PATH  = Path.home() / "modis_kd490_dbo_8d_ckpt.pkl"  # add your path
CKPT_EVERY = 100
CACHE_DERIVED = False
CACHE_PATH    = Path.home() / "modis_kd490_dbo_8d_products.pkl"  # add your path

DBO_BOXES = {
    "DBO1": [(63.769462,-172.312422),(61.945988,-172.186927),(61.846516,-175.790978),(63.662718,-176.147111)],
    "DBO2": [(65.107747,-170.865176),(65.097001,-167.859549),(64.481828,-167.907758),(64.493658,-170.840669)],
    "DBO3": [(66.785880,-171.330008),(68.608869,-171.419334),(68.572412,-166.481029),(66.752281,-166.755808)],
    "DBO4": [(71.994000,-164.226000),(71.801000,-159.729000),(70.641000,-160.305000),(70.821000,-164.556000)],
    "DBO5": [(71.807730,-158.537691),(71.635583,-155.931220),(71.110941,-156.307277),(71.278277,-158.847920)],
    "DBO6": [(71.850000,-150.108000),(70.890000,-150.976000),(71.159000,-153.865000),(72.119000,-152.980000)],
    "DBO7": [(70.868000,-141.528000),(69.946000,-142.679000),(70.333000,-145.439000),(71.253000,-144.256000)],
    "DBO8": [(70.834000,-125.155000),(70.049000,-126.997000),(70.664000,-129.354000),(71.442000,-127.431000)],
}
BOX_PATHS = {n: MplPath([(lo, la) for la, lo in c] + [(c[0][1], c[0][0])]) for n, c in DBO_BOXES.items()}
data_crs = ccrs.LambertAzimuthalEqualArea(central_latitude=90, central_longitude=0)
disp     = ccrs.LambertAzimuthalEqualArea(central_latitude=90, central_longitude=-152)
x0, y0, x1, y1 = dbo_area.area_extent
try: import cmocean; OC_CMAP = cmocean.cm.turbid
except ImportError: OC_CMAP = "viridis"
print(f"{OC_SHORT} {GRANULE_PAT} | {START_DATE} -> {END_DATE} | grid {dbo_area.shape} | streaming (no disk)")

MODISA_L3m_KD *.8D.KD.Kd_490.4km.nc | 2002-07-01 -> 2026-06-29 | grid (845, 906) | streaming (no disk)


## 3. Auth + search the monthly granules

In [3]:
auth = earthaccess.login(persist=True)

def _search(s, e, limit=2000):
    for k in range(5):
        try:
            return earthaccess.search_data(concept_id=OC_CONCEPT, granule_name=GRANULE_PAT,
                                           temporal=(s, e), count=limit)
        except Exception as ex:
            print("  retry:", str(ex)[:80]); _time.sleep(2*(k+1))
    raise RuntimeError("CMR search failed")

def granule_date(g):
    d = g.data_links()[0].rsplit("/", 1)[-1].split(".")[1].split("_")[0]
    return date(int(d[:4]), int(d[4:6]), int(d[6:8]))

all_granules = _search(START_DATE.isoformat(), END_DATE.isoformat())
granules_by_period = {}
for g in all_granules:
    granules_by_period.setdefault(granule_date(g), g)
print(f"{len(all_granules)} monthly granules | {len(granules_by_period)} unique months "
      f"({min(granules_by_period)} -> {max(granules_by_period)})" if granules_by_period else "no granules")
if all_granules:
    print("example:", all_granules[0].data_links()[0].rsplit('/',1)[-1])

1076 monthly granules | 1076 unique months (2002-07-04 -> 2025-11-17)
example: AQUA_MODIS.20020704_20020711.L3m.8D.KD.Kd_490.4km.nc


## 4. Reader - stream one month -> subset DBO bbox -> regrid onto EASE-2

In [5]:
def _read_oc(f):
    with xr.open_dataset(f, engine="h5netcdf") as ds:
        lat = np.asarray(ds[LATN].values, float); lon = np.asarray(ds[LONN].values, float)
        lon0, la0, lon1, la1 = DBO_BBOX
        jla = np.where((lat >= la0) & (lat <= la1))[0]; ilo = np.where((lon >= lon0) & (lon <= lon1))[0]
        if jla.size == 0 or ilo.size == 0: return None
        sub = ds[VAR].isel({LATN: slice(int(jla[0]), int(jla[-1]) + 1),
                            LONN: slice(int(ilo[0]), int(ilo[-1]) + 1)})
        v  = np.asarray(sub.values, float).squeeze()
        la = np.asarray(sub[LATN].values, float); lo = np.asarray(sub[LONN].values, float)
    v = np.where(v > 0, v, np.nan)          # strictly positive; drop fills / non-positive
    return la, lo, v

def grid_to_area(lat, lon, val, area):
    m = np.isfinite(val)
    if not m.any():                          # polar-night / fully-clouded month -> no obs
        return np.full(area.shape, np.nan, np.float32)
    swath = geometry.SwathDefinition(lons=lon[m], lats=lat[m])
    return resample_nearest(swath, val[m], area, radius_of_influence=ROI,
                            fill_value=np.nan, reduce_data=False).astype(np.float32)

def project_month(granule, area):
    f = earthaccess.open([granule], show_progress=False)[0]
    try: out = _read_oc(f)
    finally:
        try: f.close()
        except Exception: pass
    if out is None: return np.full(area.shape, np.nan, np.float32)
    la, lo, v = out
    lon2d, lat2d = np.meshgrid(lo, la)
    return grid_to_area(lat2d, lon2d, v, area)

## 5. DBO box masks on the EASE-2 grid

In [6]:
lons2d, lats2d = dbo_area.get_lonlats()
grid_pts = np.column_stack([lons2d.ravel(), lats2d.ravel()])
box_masks = {name: MplPath([(lo,la) for la,lo in corners] + [(corners[0][1], corners[0][0])])
                       .contains_points(grid_pts).reshape(lons2d.shape)
             for name, corners in DBO_BOXES.items()}
for name, m in box_masks.items():
    print(f"  {name}: {int(m.sum()):>6,} grid cells")

  DBO1:  2,412 grid cells
  DBO2:    605 grid cells
  DBO3:  2,561 grid cells
  DBO4:  1,293 grid cells
  DBO5:    351 grid cells
  DBO6:    739 grid cells
  DBO7:    773 grid cells
  DBO8:    760 grid cells


## 6. Streaming pass (parallel, checkpointed) - build everything

In [7]:
import pickle
def _have_products():
    return all(k in globals() and globals()[k] is not None for k in ("box_ts","mean_map","annual_oc"))
ny, nx = dbo_area.shape

if CACHE_DERIVED and CACHE_PATH.exists() and not _have_products():
    _d = pickle.load(open(CACHE_PATH, "rb"))
    box_ts, mean_map, annual_oc = _d["box_ts"], _d["mean_map"], _d["annual_oc"]
    print("loaded finished products - no streaming.")

if _have_products():
    print("products already in memory - skipping the stream.")
else:
    if CHECKPOINT and CKPT_PATH.exists():
        _ck = pickle.load(open(CKPT_PATH, "rb"))
        mean_sum, mean_cnt, ann, box_rows = _ck["mean_sum"], _ck["mean_cnt"], _ck["ann"], _ck["box_rows"]
        print(f"resumed: {len(box_rows)} months already folded.")
    else:
        mean_sum = np.zeros((ny, nx), np.float32); mean_cnt = np.zeros((ny, nx), np.float32)
        ann = {}; box_rows = {}

    def _stream_one(item):
        d, g = item
        try: return d, project_month(g, dbo_area), None
        except Exception as e: return d, None, f"{type(e).__name__}: {str(e)[:70]}"

    def _fold(d, arr):
        m = np.isfinite(arr)
        mean_sum[m] += arr[m]; mean_cnt[m] += 1
        s = ann.setdefault(d.year, [np.zeros((ny, nx), np.float32), np.zeros((ny, nx), np.float32)])
        s[0][m] += arr[m]; s[1][m] += 1
        bm = {}
        for n, mask in box_masks.items():
            v = arr[mask & m]; bm[n] = float(np.mean(v)) if v.size else np.nan
        box_rows[d] = bm

    def _save_ckpt():
        if not CHECKPOINT: return
        CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
        tmp = CKPT_PATH.with_suffix(".tmp")
        pickle.dump({"mean_sum": mean_sum, "mean_cnt": mean_cnt, "ann": ann, "box_rows": box_rows}, open(tmp, "wb"))
        tmp.replace(CKPT_PATH)

    remaining = [(d, g) for d, g in sorted(granules_by_period.items()) if d not in box_rows]
    src = iter(remaining); MAXQ = max(2, N_WORKERS*2); t0 = _time.time(); n_ok = n_fail = since = 0
    print(f"{len(box_rows)} done | streaming {len(remaining)} more...")
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        inflight = set()
        for _ in range(MAXQ):
            try: inflight.add(ex.submit(_stream_one, next(src)))
            except StopIteration: break
        while inflight:
            done, inflight = wait(inflight, return_when=FIRST_COMPLETED)
            for fut in done:
                d, arr, err = fut.result()
                if arr is None:
                    n_fail += 1
                    if n_fail <= 10: print(f"  {d}  X {err}")
                else:
                    _fold(d, arr); n_ok += 1; since += 1
                    if n_ok % 50 == 0: print(f"  {n_ok} ok / {n_fail} failed  ({(_time.time()-t0)/60:.1f} min)")
                    if since >= CKPT_EVERY: _save_ckpt(); since = 0
                try: inflight.add(ex.submit(_stream_one, next(src)))
                except StopIteration: pass
            del done
    _save_ckpt()

    dates = sorted(box_rows); tcoord = np.array([np.datetime64(d.isoformat()) for d in dates])
    box_ts = {n: xr.DataArray(np.array([box_rows[d][n] for d in dates]), dims="time", coords={"time": tcoord})
              for n in box_masks}
    mean_map = np.where(mean_cnt > 0, mean_sum / mean_cnt, np.nan)
    _years = sorted(ann)
    annual_oc = xr.DataArray(np.stack([np.where(ann[y][1] > 0, ann[y][0]/ann[y][1], np.nan) for y in _years]),
                             dims=("year","y","x"), coords={"year": _years})
    gc.collect()
    print(f"\nfolded {len(box_rows)} months -> box_ts, mean_map, {len(_years)} annual frames.")

if CACHE_DERIVED and not CACHE_PATH.exists():
    pickle.dump({"box_ts": box_ts, "mean_map": mean_map, "annual_oc": annual_oc}, open(CACHE_PATH, "wb"))
    print("saved finished products.")

0 done | streaming 1076 more...
  50 ok / 0 failed  (0.2 min)
  100 ok / 0 failed  (0.4 min)
  150 ok / 0 failed  (0.5 min)
  200 ok / 0 failed  (0.7 min)
  250 ok / 0 failed  (0.8 min)
  300 ok / 0 failed  (1.0 min)
  350 ok / 0 failed  (1.2 min)
  400 ok / 0 failed  (1.3 min)
  450 ok / 0 failed  (1.5 min)
  500 ok / 0 failed  (1.6 min)
  550 ok / 0 failed  (1.8 min)
  600 ok / 0 failed  (2.0 min)
  650 ok / 0 failed  (2.2 min)
  700 ok / 0 failed  (2.3 min)
  750 ok / 0 failed  (2.5 min)
  800 ok / 0 failed  (2.7 min)
  850 ok / 0 failed  (2.8 min)
  900 ok / 0 failed  (3.1 min)
  950 ok / 0 failed  (3.3 min)
  1000 ok / 0 failed  (3.5 min)
  1050 ok / 0 failed  (3.7 min)

folded 1076 months -> box_ts, mean_map, 24 annual frames.


## 7. Trend helpers

In [8]:
def _lag1(x):
    x = np.asarray(x, float); a, b = x[1:], x[:-1]; v = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[v], b[v])[0, 1]) if v.sum() >= 5 else 0.0
def mk(x, y, minn=24):
    v = np.isfinite(x) & np.isfinite(y); x, y = x[v], y[v]; n = len(x)
    if n < minn: return dict(slope=np.nan, intercept=np.nan, p=np.nan, p_naive=np.nan, n=n)
    slope, intercept, _, _ = theilslopes(y, x)
    tau, p_naive = kendalltau(x, y)
    sl, ic = np.polyfit(x, y, 1); phi = _lag1(y - (sl*x + ic))
    ne = n*(1-phi)/(1+phi) if (abs(phi) < 1 and phi != 0) else n
    p = 2*(1 - norm.cdf(abs(tau/np.sqrt(2*(2*ne+5)/(9*ne*(ne-1)))))) if ne > 4 else np.nan
    return dict(slope=slope, intercept=intercept, p=p, p_naive=p_naive, n=n)
def _stars(p):
    return ("***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns") if np.isfinite(p) else "n/a"
print("helpers ready")

helpers ready


## 8. Per-DBO-box monthly Kd(490) + month-of-year anomaly trend (Theil-Sen + AR(1) MK)

8-day series is resampled to monthly, then deseasonalized by month-of-year and trended. Reported as % of the box mean per decade.
*log10 variant:* wrap `s` in `np.log10(s)` before the anomaly step to trend relative change.

In [9]:
names = [n for n in DBO_BOXES if n in box_ts and np.isfinite(box_ts[n]).sum() > 24]
fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True); axes = axes.ravel()
for ax, name in zip(axes, names):
    s = box_ts[name].resample(time="1MS").mean(skipna=True)
    moy = s["time"].dt.month
    anom = s.assign_coords(moy=("time", moy.values)).groupby("moy") - \
           s.assign_coords(moy=("time", moy.values)).groupby("moy").mean()
    t = s["time"].values; dec = pd.to_datetime(t); dec = np.asarray(dec.year + (dec.month-0.5)/12.0, float)
    r = mk(dec, anom.values); base = float(np.nanmean(s.values))
    ax.plot(t, s.values, "-", color="#b9a", lw=0.5, alpha=0.6)
    ax.plot(t, anom.values, "-", color="#843", lw=0.7)
    ax.axhline(0, color="gray", lw=0.5)
    if np.isfinite(r["slope"]):
        ax.plot(t, r["intercept"] + r["slope"]*dec, "-", color="#1f4ec7", lw=1.6)
        pct = 100*r["slope"]*10/base if base else np.nan
        lab = f"{r['slope']*10:+.1e}/dec ({pct:+.1f}%/dec) {_stars(r['p'])}"
    else: lab = f"(n={r['n']}, insufficient)"
    ax.set_title(f"{name}  {lab}", fontsize=8.5, loc="left"); ax.grid(alpha=0.25)
    ax.xaxis.set_major_locator(mdates.YearLocator(base=5)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    for sp in ("top","right"): ax.spines[sp].set_visible(False)
for ax in axes[len(names):]: ax.axis("off")
for ax in axes[::4]: ax.set_ylabel(f"{LABEL} ({UNITS})")
fig.suptitle(f"MODIS-Aqua {LABEL} per DBO box - monthly (light) + deseasonalized anomaly + Theil-Sen/AR(1) MK trend",
             fontsize=13, y=1.0)
plt.tight_layout(); plt.show()

## 9. Per-box trend table (deseasonalized monthly anomaly)

In [10]:
rows = []
for name in names:
    s = box_ts[name].resample(time="1MS").mean(skipna=True); moy = s["time"].dt.month
    anom = s.assign_coords(moy=("time", moy.values)).groupby("moy") - \
           s.assign_coords(moy=("time", moy.values)).groupby("moy").mean()
    t = pd.to_datetime(s["time"].values); dec = np.asarray(t.year + (t.month-0.5)/12.0, float)
    r = mk(dec, anom.values); base = float(np.nanmean(s.values))
    rows.append((name, base, r["slope"]*10, 100*r["slope"]*10/base if base else np.nan,
                 r["p_naive"], r["p"], _stars(r["p"]), r["n"]))
tbl = pd.DataFrame(rows, columns=["box","mean","trend(/dec)","trend(%/dec)","p_naive","p_AR1","sig","n_months"]).set_index("box")
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
print(f"MODIS {LABEL} trend per DBO box (deseasonalized monthly, Theil-Sen + MK):\n"); print(tbl)

MODIS Kd(490) trend per DBO box (deseasonalized monthly, Theil-Sen + MK):

        mean  trend(/dec)  trend(%/dec)  p_naive    p_AR1 sig  n_months
box                                                                    
DBO1  0.1209      0.01006         8.316 0.001521 0.001446  **       183
DBO2  0.2104      0.02541         12.08 0.007021 0.008276  **       150
DBO3  0.1536      0.01103         7.179 0.004482  0.02257   *       152
DBO4 0.09886     0.001473          1.49   0.4911   0.5192  ns       132
DBO5  0.1297     0.002856         2.202   0.5234   0.5562  ns       141
DBO6  0.1291      0.01143         8.854  0.03257  0.06482  ns       112
DBO7  0.1416      0.01413         9.977   0.0808   0.2432  ns       114
DBO8   0.119     0.004264         3.583   0.1424    0.329  ns       146
